# 07 — Feature Engineering for Next-Basket Prediction

## 1. Objective and Prediction Framework

The objective of this notebook is to construct a machine-learning-ready dataset for predicting whether a customer will reorder a candidate product in their next order.

Each observation in the final modeling dataset represents a:

**Customer × Product × Target Order**

relationship.

The prediction target will be:

- `1` — the product appears in the customer's target order
- `0` — the product does not appear in the customer's target order

To prevent data leakage, all features must be calculated exclusively from orders that occurred before the target order.

The feature engineering framework will include:

1. Customer behavioral features
2. Product-level features
3. Customer-product interaction features
4. Recency and reorder-cycle features
5. Purchase momentum and streak features
6. Department and aisle affinity features
7. Temporal behavior features
8. Basket and co-purchase relationship features

The resulting feature table will later be used for next-basket prediction and product recommendation.

## 2. Historical and Target Order Definition

The Instacart dataset separates customer orders into `prior`, `train`, and `test` sets.

For this modeling problem:

- `prior` orders are used exclusively as historical information for feature engineering.
- `train` orders are used as the target orders because their product contents are available.
- `test` orders are excluded because their product contents are not provided.

For each customer included in the modeling population:

**Historical orders = all `prior` orders**

**Target order = the customer's `train` order**

All customer, product, customer-product, temporal, and behavioral features must therefore be calculated exclusively from `prior` orders.

The `train` order will only be used afterward to construct the prediction target.

This separation prevents information from the target basket from leaking into the feature engineering process.

### 2.1 — Load the Cleaned Transaction Tables

We load the cleaned order and product-transaction tables that will be used to build the historical and target datasets.

In [0]:
from pyspark.sql import functions as F

orders_df = spark.table("workspace.cleaned_data.orders")
order_products_df = spark.table("workspace.cleaned_data.order_products")

display(orders_df.limit(10))

order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,order_time_period,days_since_prior_order,is_first_order,has_prior_order,is_final_order,_source_file,_imported_data_ingested_at,_cleaned_data_processed_at
2539329,1,prior,1,2,8,morning,null,true,false,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
2398795,1,prior,2,3,7,morning,15.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
473747,1,prior,3,3,12,afternoon,21.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
2254736,1,prior,4,4,7,morning,29.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
431534,1,prior,5,4,15,afternoon,28.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
3367565,1,prior,6,2,7,morning,19.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
550135,1,prior,7,1,9,morning,20.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
3108588,1,prior,8,1,14,afternoon,14.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
2295261,1,prior,9,1,16,afternoon,0.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z
2550362,1,prior,10,4,8,morning,30.0,false,true,false,orders.csv,2026-07-26T23:04:59.486Z,2026-08-13T15:45:59.188Z


### 2.2 — Create the Target Orders

We select each customer's `train` order and rename its columns clearly. This order will be used later as the basket we want to predict.

In [0]:
target_orders_df = (
    orders_df
    .filter(F.col("eval_set") == "train")
    .select(
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .withColumnRenamed("order_id", "target_order_id")
    .withColumnRenamed("order_number", "target_order_number")
    .withColumnRenamed("order_dow", "target_order_dow")
    .withColumnRenamed("order_hour_of_day", "target_order_hour")
    .withColumnRenamed(
        "days_since_prior_order",
        "target_days_since_prior_order"
    )
)

display(target_orders_df.limit(10))

target_order_id,user_id,target_order_number,target_order_dow,target_order_hour,target_days_since_prior_order
1187899,1,11,4,8,14.0
1492625,2,15,1,11,30.0
2196797,5,5,0,11,6.0
525192,7,21,2,11,6.0
880375,8,4,1,14,10.0
1094988,9,4,6,10,30.0
1822501,10,6,0,19,30.0
1827621,13,13,0,21,8.0
2316178,14,14,2,19,11.0
2180313,17,41,3,10,30.0


### 2.3 — Verify One Target Order per Customer

We check that each customer has exactly one target order. If the number of target orders equals the number of unique customers, the setup is correct.

In [0]:
target_order_check_df = (
    target_orders_df
    .agg(
        F.count("*").alias("target_orders"),
        F.countDistinct("user_id").alias("unique_customers")
    )
)

display(target_order_check_df)

target_orders,unique_customers
131209,131209


### 2.4 — Create the Historical Orders

We keep only the `prior` orders of customers who have a target order. These orders represent the customer history that will be used to create features.

In [0]:
historical_orders_df = (
    orders_df
    .filter(F.col("eval_set") == "prior")
    .join(
        target_orders_df.select("user_id"),
        on="user_id",
        how="inner"
    )
    .select(
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
)

display(historical_orders_df.limit(10))

order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
2539329,1,1,2,8,null
2398795,1,2,3,7,15.0
473747,1,3,3,12,21.0
2254736,1,4,4,7,29.0
431534,1,5,4,15,28.0
3367565,1,6,2,7,19.0
550135,1,7,1,9,20.0
3108588,1,8,1,14,14.0
2295261,1,9,1,16,0.0
2550362,1,10,4,8,30.0


### 2.5 — Create Historical Product Transactions

We connect each historical order with the products that were purchased in it. This gives us the customer-product history that will be used to create features.

In [0]:
historical_transactions_df = (
    historical_orders_df
    .join(
        order_products_df.select(
            "order_id",
            "product_id",
            "add_to_cart_order",
            "reordered"
        ),
        on="order_id",
        how="inner"
    )
    .select(
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "product_id",
        "add_to_cart_order",
        "reordered"
    )
)

display(historical_transactions_df.limit(10))

order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
2,202279,3,5,9,8.0,33120,1,1
2,202279,3,5,9,8.0,28985,2,1
2,202279,3,5,9,8.0,9327,3,0
2,202279,3,5,9,8.0,45918,4,1
2,202279,3,5,9,8.0,30035,5,0
2,202279,3,5,9,8.0,17794,6,1
2,202279,3,5,9,8.0,40141,7,1
2,202279,3,5,9,8.0,1819,8,1
2,202279,3,5,9,8.0,43668,9,0
3,205970,16,5,17,12.0,33754,1,1


### 2.6 — Create the Target Basket Products

We extract the products that actually appeared in each customer's `train` order. These products will later be used only to create the prediction label.

In [0]:
target_products_df = (
    target_orders_df
    .select(
        "target_order_id",
        "user_id"
    )
    .join(
        order_products_df.select(
            F.col("order_id").alias("target_order_id"),
            "product_id"
        ),
        on="target_order_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

display(target_products_df.limit(10))

user_id,target_order_id,product_id
112108,1,49302
112108,1,11109
112108,1,10246
112108,1,49683
112108,1,43633
112108,1,13176
112108,1,47209
112108,1,22035
79431,36,39612
79431,36,19660


## 3. Candidate Generation

### 3.1 — Create Previously Purchased Product Candidates

For each customer, we keep the unique products they purchased in their historical orders. These products become the candidates that the model will evaluate for the next basket.

In [0]:
candidate_products_df = (
    historical_transactions_df
    .select(
        "user_id",
        "product_id"
    )
    .distinct()
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

display(candidate_products_df.limit(10))

user_id,target_order_id,product_id
155476,861321,38770
96385,1797479,24933
14264,2880532,43955
92123,2136310,9124
21630,57725,41453
67140,2901738,8214
44598,2031618,45437
201679,2310177,13176
34076,2244102,31981
84876,2179086,28050


### 3.2 — Create the Reorder Target

We check whether each candidate product appears in the customer's target basket. Products that appear receive `1`, and products that do not appear receive `0`.

In [0]:
target_labels_df = (
    target_products_df
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .withColumn("target_reordered", F.lit(1))
)

model_base_df = (
    candidate_products_df
    .join(
        target_labels_df,
        on=["user_id", "target_order_id", "product_id"],
        how="left"
    )
    .fillna({"target_reordered": 0})
)

display(model_base_df.limit(10))

user_id,target_order_id,product_id,target_reordered
156122,1277092,20914,1
135442,2194248,3464,0
30822,1360042,46941,0
96594,647971,9387,0
86865,2133756,7948,0
91891,3419917,37371,0
122266,1562056,23339,0
54878,1245991,1181,0
180454,3189128,35333,0
153628,1089432,23888,0


### 3.3 — Check the Target Distribution

We count how many candidate products were reordered (`1`) and not reordered (`0`). This helps us see whether the prediction target is balanced or imbalanced.

In [0]:
label_distribution_df = (
    model_base_df
    .groupBy("target_reordered")
    .agg(
        F.count("*").alias("candidate_count")
    )
    .orderBy("target_reordered")
)

total_candidates = model_base_df.count()

label_distribution_df = (
    label_distribution_df
    .withColumn(
        "percentage",
        F.round(
            F.col("candidate_count") / F.lit(total_candidates) * 100,
            2
        )
    )
)

display(label_distribution_df)

target_reordered,candidate_count,percentage
0,7645837,90.22
1,828824,9.78


### 3.4 — Measure Candidate Coverage

We check what percentage of products in the real target baskets were already purchased before and therefore appear in our candidate list. This tells us how much of the next basket our current candidate strategy can potentially predict.

In [0]:
target_product_count_df = (
    target_products_df
    .select("user_id", "target_order_id", "product_id")
    .distinct()
    .agg(
        F.count("*").alias("total_target_products")
    )
)

covered_target_product_count_df = (
    model_base_df
    .filter(F.col("target_reordered") == 1)
    .agg(
        F.count("*").alias("covered_target_products")
    )
)

candidate_coverage_df = (
    target_product_count_df
    .crossJoin(covered_target_product_count_df)
    .withColumn(
        "coverage_percentage",
        F.round(
            F.col("covered_target_products")
            / F.col("total_target_products")
            * 100,
            2
        )
    )
)

display(candidate_coverage_df)

total_target_products,covered_target_products,coverage_percentage
1384617,828824,59.86


1,384,617 products appear in the real target baskets.
828,824 of them were bought before.
Our current candidate strategy covers 59.86% of the real next-basket products.

So about 40.14% are new-to-customer products. We’ll keep this as our baseline and later improve candidate generation with aisle affinity and co-purchase signals.

## 4. Customer Behavioral Features

### 4.1 — Create Basic Customer Features

We summarize each customer's historical shopping behavior, such as number of orders, basket size, reorder rate, and purchase variety.

In [0]:
customer_features_df = (
    historical_transactions_df
    .groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("customer_prior_orders"),
        F.count("product_id").alias("customer_total_products"),
        F.countDistinct("product_id").alias("customer_unique_products"),
        F.round(
            F.count("product_id") / F.countDistinct("order_id"),
            2
        ).alias("customer_avg_basket_size"),
        F.round(
            F.avg("reordered"),
            4
        ).alias("customer_reorder_rate")
    )
)

display(customer_features_df.limit(10))

user_id,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate
177763,8,24,19,3.0,0.2083
147726,12,238,113,19.83,0.5252
50762,24,608,128,25.33,0.7895
64362,9,100,73,11.11,0.27
143392,10,41,21,4.1,0.4878
165080,24,408,141,17.0,0.6544
104834,40,344,183,8.6,0.468
134466,13,129,93,9.92,0.2791
191785,15,161,62,10.73,0.6149
184764,34,739,234,21.74,0.6834


### 4.2 — Create Customer Ordering Rhythm Features

We measure how often and how regularly each customer places orders. These features help distinguish routine shoppers from customers with more irregular shopping habits.

In [0]:
customer_rhythm_features_df = (
    historical_orders_df
    .groupBy("user_id")
    .agg(
        F.round(
            F.avg("days_since_prior_order"),
            2
        ).alias("customer_avg_days_between_orders"),

        F.round(
            F.stddev_pop("days_since_prior_order"),
            2
        ).alias("customer_std_days_between_orders"),

        F.round(
            F.avg("order_hour_of_day"),
            2
        ).alias("customer_avg_order_hour"),

        F.countDistinct("order_dow")
        .alias("customer_active_days_of_week")
    )
)

display(customer_rhythm_features_df.limit(10))

user_id,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week
93,13.31,11.93,11.29,6
218,15.8,7.44,13.0,3
261,3.71,2.05,15.25,5
282,26.0,4.0,14.67,2
323,2.43,2.14,12.94,6
406,21.0,12.73,12.5,3
605,16.25,8.17,13.0,4
677,20.0,10.39,13.0,3
1003,9.0,3.25,17.0,2
1021,9.74,7.98,13.34,6


### 4.3 — Create Recent Basket Trend Features

We compare a customer's recent basket sizes with their usual basket size. This helps detect whether they are currently buying more or fewer products than normal.

In [0]:
from pyspark.sql.window import Window

basket_per_order_df = (
    historical_transactions_df
    .groupBy(
        "user_id",
        "order_id",
        "order_number"
    )
    .agg(
        F.count("product_id").alias("basket_size")
    )
)

recent_order_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.desc("order_number"))
)

ranked_baskets_df = (
    basket_per_order_df
    .withColumn(
        "recent_order_rank",
        F.row_number().over(recent_order_window)
    )
)

customer_recent_basket_df = (
    ranked_baskets_df
    .groupBy("user_id")
    .agg(
        F.max(
            F.when(
                F.col("recent_order_rank") == 1,
                F.col("basket_size")
            )
        ).alias("customer_last_basket_size"),

        F.round(
            F.avg(
                F.when(
                    F.col("recent_order_rank") <= 3,
                    F.col("basket_size")
                )
            ),
            2
        ).alias("customer_avg_last_3_basket_size")
    )
)

customer_basket_trend_df = (
    customer_recent_basket_df
    .join(
        customer_features_df.select(
            "user_id",
            "customer_avg_basket_size"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "customer_basket_size_trend",
        F.round(
            F.col("customer_avg_last_3_basket_size")
            - F.col("customer_avg_basket_size"),
            2
        )
    )
    .select(
        "user_id",
        "customer_last_basket_size",
        "customer_avg_last_3_basket_size",
        "customer_basket_size_trend"
    )
)

display(customer_basket_trend_df.limit(10))

user_id,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend
70421,5,3.0,-0.45
50436,16,13.0,6.85
76380,18,13.0,0.0
18816,1,1.0,-0.26
193551,11,10.0,1.43
80583,4,6.0,2.44
74886,15,17.33,4.97
132815,2,5.0,-6.57
160964,6,6.67,-4.78
186338,40,21.67,8.15


For example, a positive value such as 6.85 means the customer's recent baskets are much larger than their historical average, while a negative value means recent baskets are smaller.

### 4.4 — Create Customer Shopping Routine Features

We identify when each customer usually shops and how consistent that routine is. Customers with strong habits may be easier to predict.

In [0]:
# Most common shopping day for each customer
customer_day_counts_df = (
    historical_orders_df
    .groupBy("user_id", "order_dow")
    .agg(
        F.count("*").alias("day_order_count")
    )
)

day_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("day_order_count"),
        F.asc("order_dow")
    )
)

customer_preferred_day_df = (
    customer_day_counts_df
    .withColumn(
        "day_rank",
        F.row_number().over(day_window)
    )
    .filter(F.col("day_rank") == 1)
)

# Most common shopping hour for each customer
customer_hour_counts_df = (
    historical_orders_df
    .groupBy("user_id", "order_hour_of_day")
    .agg(
        F.count("*").alias("hour_order_count")
    )
)

hour_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("hour_order_count"),
        F.asc("order_hour_of_day")
    )
)

customer_preferred_hour_df = (
    customer_hour_counts_df
    .withColumn(
        "hour_rank",
        F.row_number().over(hour_window)
    )
    .filter(F.col("hour_rank") == 1)
)

customer_routine_features_df = (
    customer_preferred_day_df
    .join(
        customer_preferred_hour_df,
        on="user_id",
        how="inner"
    )
    .join(
        customer_features_df.select(
            "user_id",
            "customer_prior_orders"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "customer_preferred_day_share",
        F.round(
            F.col("day_order_count") /
            F.col("customer_prior_orders"),
            4
        )
    )
    .withColumn(
        "customer_preferred_hour_share",
        F.round(
            F.col("hour_order_count") /
            F.col("customer_prior_orders"),
            4
        )
    )
    .select(
        "user_id",
        F.col("order_dow").alias("customer_preferred_dow"),
        "customer_preferred_day_share",
        F.col("order_hour_of_day").alias("customer_preferred_hour"),
        "customer_preferred_hour_share"
    )
)

display(customer_routine_features_df.limit(10))

user_id,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share
152996,0,0.1695,18,0.1356
183485,0,0.3077,15,0.2308
56444,1,0.3077,9,0.3077
26705,1,0.2941,17,0.2353
183187,0,0.5128,10,0.1538
159187,0,0.4286,11,0.1429
127653,2,0.2727,15,0.2727
117292,5,0.4,20,0.2
110621,5,0.2222,16,0.2593
105805,3,0.8261,8,0.2609


### 4.5 — Combine Customer Features

We combine all customer-level features into one table so they can be joined easily with the final modeling dataset later.

In [0]:
all_customer_features_df = (
    customer_features_df
    .join(
        customer_rhythm_features_df,
        on="user_id",
        how="left"
    )
    .join(
        customer_basket_trend_df,
        on="user_id",
        how="left"
    )
    .join(
        customer_routine_features_df,
        on="user_id",
        how="left"
    )
)

display(all_customer_features_df.limit(10))

user_id,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share
125432,39,122,41,3.13,0.6639,7.29,6.57,15.97,7,4,4.67,1.54,5,0.2308,17,0.2051
88497,14,224,107,16.0,0.5223,19.46,10.11,15.93,5,29,23.33,7.33,0,0.5,18,0.3571
197841,45,1139,269,25.31,0.7638,7.82,4.02,14.38,6,21,30.33,5.02,6,0.4667,8,0.1778
199398,37,608,251,16.43,0.5872,7.56,5.62,13.78,7,10,16.0,-0.43,0,0.2973,20,0.1622
155877,12,205,56,17.08,0.7268,12.18,8.26,13.08,5,24,22.33,5.25,5,0.3333,8,0.1667
131680,5,106,56,21.2,0.4717,10.75,2.77,13.8,3,17,26.67,5.47,6,0.6,9,0.2
33017,12,68,42,5.67,0.3824,15.55,10.97,17.0,7,7,4.67,-1.0,2,0.3333,18,0.25
144401,29,223,85,7.69,0.6188,8.64,6.18,14.59,7,13,13.33,5.64,0,0.2759,20,0.1724
182621,37,296,144,8.0,0.5135,9.19,7.28,15.11,7,5,6.0,-2.0,0,0.2432,18,0.1622
168277,9,52,37,5.78,0.2885,19.75,8.61,14.33,5,13,7.67,1.89,1,0.4444,16,0.3333


## 5. Product-Level Features

### 5.1 — Create Basic Product Features

We summarize how each product behaves historically, including its popularity, reorder rate, customer reach, and usual cart position.

product_purchase_count       → how often the product was purchased

product_unique_customers     → how many customers bought it

product_reorder_rate         → how often it was a repeat purchase

product_avg_cart_position    → where it is usually added in the basket

In [0]:
product_features_df = (
    historical_transactions_df
    .groupBy("product_id")
    .agg(
        F.count("*")
        .alias("product_purchase_count"),

        F.countDistinct("user_id")
        .alias("product_unique_customers"),

        F.round(
            F.avg("reordered"),
            4
        ).alias("product_reorder_rate"),

        F.round(
            F.avg("add_to_cart_order"),
            2
        ).alias("product_avg_cart_position")
    )
)

display(product_features_df.limit(10))

product_id,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position
13176,239739,40493,0.8311,5.12
41970,882,408,0.5374,8.47
35921,8909,2631,0.7047,6.66
41623,215,133,0.3814,9.35
34688,1629,517,0.6826,7.11
36051,1090,754,0.3083,4.98
28473,1000,504,0.496,8.8
16283,3756,1368,0.6358,4.35
7533,2515,794,0.6843,8.9
21847,2638,1415,0.4636,9.53


### 5.2 — Add Product Category and Loyalty Features

We add each product's aisle and department and measure how repeatedly customers buy it. This helps distinguish staple products from occasional purchases.

product_purchases_per_customer → how repeatedly buyers purchase the product

product_order_share            → how common the product is across all historical orders

In [0]:
products_df = (
    spark.table("workspace.cleaned_data.products")
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    )
)

total_historical_orders = historical_orders_df.count()

product_extended_features_df = (
    product_features_df
    .join(
        products_df,
        on="product_id",
        how="left"
    )
    .withColumn(
        "product_purchases_per_customer",
        F.round(
            F.col("product_purchase_count")
            / F.col("product_unique_customers"),
            2
        )
    )
    .withColumn(
        "product_order_share",
        F.round(
            F.col("product_purchase_count")
            / F.lit(total_historical_orders),
            6
        )
    )
)

display(product_extended_features_df.limit(10))

product_id,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share
32205,301,211,0.299,9.6,Fluoride-Free Antiplaque & Whitening Spearmint Toothpaste,20,11,1.43,1.47E-4
43154,11441,2505,0.7811,4.18,Sparkling Mineral Water,115,7,4.57,0.005588
11193,2061,737,0.6424,9.1,Ancient Grain Blueberry Hemp Granola,57,14,2.8,0.001007
40311,982,406,0.5866,7.58,Organic Mixed Baby Kale Salad,123,4,2.42,4.8E-4
39329,80,28,0.65,9.51,Dr. Better Soda,77,7,2.86,3.9E-5
30135,2379,826,0.6528,9.8,Cheddar Cheese Burrito,38,1,2.88,0.001162
17896,537,238,0.5568,8.43,Sugar Free Lemon Lime Gelatin Snacks,71,16,2.26,2.62E-4
23767,147,72,0.5102,4.87,Imported Light Beer,27,5,2.04,7.2E-5
17389,293,197,0.3276,9.17,Jack Habanero Cheese,83,4,1.49,1.43E-4
11777,16641,5987,0.6402,8.15,Red Raspberries,123,4,2.78,0.008128


### 5.3 — Create Product Temporal Preference Features

We identify the day and hour when each product is most often purchased. These features will later help compare product habits with the customer's target-order timing.

In [0]:
# Most common purchase day for each product
product_day_counts_df = (
    historical_transactions_df
    .groupBy("product_id", "order_dow")
    .agg(
        F.count("*").alias("product_day_purchase_count")
    )
)

product_day_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.desc("product_day_purchase_count"),
        F.asc("order_dow")
    )
)

product_preferred_day_df = (
    product_day_counts_df
    .withColumn(
        "day_rank",
        F.row_number().over(product_day_window)
    )
    .filter(F.col("day_rank") == 1)
)

# Most common purchase hour for each product
product_hour_counts_df = (
    historical_transactions_df
    .groupBy("product_id", "order_hour_of_day")
    .agg(
        F.count("*").alias("product_hour_purchase_count")
    )
)

product_hour_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.desc("product_hour_purchase_count"),
        F.asc("order_hour_of_day")
    )
)

product_preferred_hour_df = (
    product_hour_counts_df
    .withColumn(
        "hour_rank",
        F.row_number().over(product_hour_window)
    )
    .filter(F.col("hour_rank") == 1)
)

product_temporal_features_df = (
    product_preferred_day_df
    .join(
        product_preferred_hour_df,
        on="product_id",
        how="inner"
    )
    .join(
        product_features_df.select(
            "product_id",
            "product_purchase_count"
        ),
        on="product_id",
        how="inner"
    )
    .withColumn(
        "product_preferred_day_share",
        F.round(
            F.col("product_day_purchase_count")
            / F.col("product_purchase_count"),
            4
        )
    )
    .withColumn(
        "product_preferred_hour_share",
        F.round(
            F.col("product_hour_purchase_count")
            / F.col("product_purchase_count"),
            4
        )
    )
    .select(
        "product_id",
        F.col("order_dow").alias("product_preferred_dow"),
        "product_preferred_day_share",
        F.col("order_hour_of_day").alias("product_preferred_hour"),
        "product_preferred_hour_share"
    )
)

display(product_temporal_features_df.limit(10))

product_id,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share
1,1,0.2413,10,0.1248
2,6,0.2,10,0.15
4,0,0.2248,14,0.1055
5,0,0.25,11,0.25
7,0,0.3333,15,0.25
8,0,0.2373,9,0.1102
9,5,0.1979,13,0.1146
10,5,0.1746,11,0.0977
11,3,0.2759,12,0.1552
13,0,0.3333,14,0.6667


### 5.4 — Combine Product Features

We combine all product-level features into one table so they can be added easily to the final modeling dataset.

In [0]:
all_product_features_df = (
    product_extended_features_df
    .join(
        product_temporal_features_df,
        on="product_id",
        how="left"
    )
)

display(all_product_features_df.limit(10))

product_id,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share
20549,3269,2093,0.3597,10.7,Organic Orzo,131,9,1.56,0.001597,0,0.2224,11,0.0918
34067,1239,777,0.3729,9.56,Organic Honey,29,13,1.59,6.05E-4,1,0.1711,10,0.0985
26047,3818,904,0.7632,7.46,Tuna Salad,1,20,4.22,0.001865,0,0.1904,10,0.1003
17601,505,264,0.4772,8.4,Peppermint Caffeine Free Herbal Tea Bags,94,7,1.91,2.47E-4,1,0.1842,10,0.1129
1710,1152,299,0.7405,5.75,Kombucha Gingerade,31,7,3.85,5.63E-4,0,0.1632,16,0.0955
39332,1794,658,0.6332,8.64,Dried Mangoes,117,19,2.73,8.76E-4,1,0.2402,10,0.1076
37993,594,497,0.1633,8.58,Coppertop AA Batteries,87,17,1.2,2.9E-4,0,0.1886,12,0.1027
20144,1378,373,0.7293,6.82,Organic Pack Peasant Bread,112,3,3.69,6.73E-4,0,0.1647,11,0.0951
25525,178,81,0.5449,10.01,Aged Artisanal Treenut Cracked Pepper Cheese,21,16,2.2,8.7E-5,0,0.2135,16,0.118
10160,11,6,0.4545,12.0,"Veggies On-the-Go, Stage 2 (6 Months and Up), Pumpkin, Zucchini & Apple Blend",92,18,1.83,5.0E-6,0,0.2727,9,0.1818


## 6. Customer-Product Interaction Features

### 6.1 — Create Basic Customer-Product Features

We measure the relationship between each customer and product, including how often the customer bought it and how important it is in their shopping history.

In [0]:
customer_product_features_df = (
    historical_transactions_df
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        F.countDistinct("order_id")
        .alias("user_product_order_count"),

        F.round(
            F.avg("reordered"),
            4
        ).alias("user_product_reorder_rate"),

        F.round(
            F.avg("add_to_cart_order"),
            2
        ).alias("user_product_avg_cart_position"),

        F.min("order_number")
        .alias("user_product_first_order"),

        F.max("order_number")
        .alias("user_product_last_order")
    )
    .join(
        customer_features_df.select(
            "user_id",
            "customer_prior_orders"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "user_product_order_share",
        F.round(
            F.col("user_product_order_count")
            / F.col("customer_prior_orders"),
            4
        )
    )
    .drop("customer_prior_orders")
)

display(customer_product_features_df.limit(10))

user_id,product_id,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share
70421,5514,9,0.8889,2.0,1,11,0.8182
70421,11520,2,0.5,1.5,1,5,0.1818
70421,12892,1,0.0,3.0,1,1,0.0909
70421,18880,2,0.5,3.5,1,4,0.1818
70421,18441,5,0.8,3.6,1,11,0.4545
70421,36178,3,0.6667,2.67,1,9,0.2727
50436,46969,7,0.8571,5.86,5,75,0.0864
50436,21405,19,0.9474,4.05,7,80,0.2346
50436,19660,38,0.9737,2.95,1,80,0.4691
76380,26856,3,0.6667,1.67,1,3,1.0


### 6.2 — Create Recency and Reorder-Cycle Features

We measure how recently each customer bought each product and compare it with their usual purchase interval. This helps identify products that may be due for reorder.

orders_since_last_product_purchase → how long since the customer last bought it

user_product_avg_order_gap         → usual interval between purchases

user_product_due_score             → current gap compared with the usual gap

≈ 1.0  → product is around its usual reorder time

< 1.0  → probably still early

> 1.0  → product may be overdue

null   → product was bought only once, so no cycle exists yet

In [0]:
# Keep one row per customer-product-order
user_product_orders_df = (
    historical_transactions_df
    .select(
        "user_id",
        "product_id",
        "order_number"
    )
    .distinct()
)

# Calculate the gap between consecutive purchases of the same product
product_cycle_window = (
    Window
    .partitionBy("user_id", "product_id")
    .orderBy("order_number")
)

user_product_gaps_df = (
    user_product_orders_df
    .withColumn(
        "previous_product_order",
        F.lag("order_number").over(product_cycle_window)
    )
    .withColumn(
        "product_order_gap",
        F.col("order_number") - F.col("previous_product_order")
    )
)

# Average purchase interval for each customer-product pair
user_product_cycle_stats_df = (
    user_product_gaps_df
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        F.round(
            F.avg("product_order_gap"),
            2
        ).alias("user_product_avg_order_gap")
    )
)

# Compare the target order with the last historical purchase
user_product_recency_features_df = (
    customer_product_features_df
    .select(
        "user_id",
        "product_id",
        "user_product_order_count",
        "user_product_last_order"
    )
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_number"
        ),
        on="user_id",
        how="inner"
    )
    .join(
        user_product_cycle_stats_df,
        on=["user_id", "product_id"],
        how="left"
    )
    .withColumn(
        "orders_since_last_product_purchase",
        F.col("target_order_number") - F.col("user_product_last_order")
    )
    .withColumn(
        "user_product_due_score",
        F.when(
            F.col("user_product_avg_order_gap").isNotNull(),
            F.round(
                F.col("orders_since_last_product_purchase")
                / F.col("user_product_avg_order_gap"),
                2
            )
        )
    )
    .select(
        "user_id",
        "product_id",
        "orders_since_last_product_purchase",
        "user_product_avg_order_gap",
        "user_product_due_score"
    )
)

display(user_product_recency_features_df.limit(10))

user_id,product_id,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score
10882,39867,2,null,null
23584,14901,1,1.13,0.88
39643,29309,5,null,null
48968,14901,4,2.0,2.0
74315,48742,10,6.4,1.56
96718,30756,8,1.0,8.0
100330,30756,38,null,null
116543,48742,18,null,null
168641,46025,41,8.33,4.92
205970,32665,2,4.0,0.5


### 6.3 — Create Recent Purchase Momentum Features

We measure whether a customer bought each product in their most recent orders. This helps detect products that are becoming part of the customer's current shopping routine.

> 0  → product is being bought more often recently

≈ 0  → recent behavior matches the customer's usual pattern

< 0  → interest in the product may be decreasing

In [0]:
user_product_recent_df = (
    historical_transactions_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_number"
        ),
        on="user_id",
        how="inner"
    )
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        F.max(
            F.when(
                F.col("order_number") == F.col("target_order_number") - 1,
                1
            ).otherwise(0)
        ).alias("bought_in_last_order"),

        F.countDistinct(
            F.when(
                F.col("order_number") >= F.col("target_order_number") - 3,
                F.col("order_id")
            )
        ).alias("purchases_last_3_orders"),

        F.countDistinct(
            F.when(
                F.col("order_number") >= F.col("target_order_number") - 5,
                F.col("order_id")
            )
        ).alias("purchases_last_5_orders")
    )
)

user_product_momentum_features_df = (
    user_product_recent_df
    .join(
        customer_features_df.select(
            "user_id",
            "customer_prior_orders"
        ),
        on="user_id",
        how="inner"
    )
    .join(
        customer_product_features_df.select(
            "user_id",
            "product_id",
            "user_product_order_share"
        ),
        on=["user_id", "product_id"],
        how="inner"
    )
    .withColumn(
        "purchase_rate_last_3_orders",
        F.round(
            F.col("purchases_last_3_orders")
            / F.least(F.col("customer_prior_orders"), F.lit(3)),
            4
        )
    )
    .withColumn(
        "purchase_rate_last_5_orders",
        F.round(
            F.col("purchases_last_5_orders")
            / F.least(F.col("customer_prior_orders"), F.lit(5)),
            4
        )
    )
    .withColumn(
        "recent_purchase_momentum",
        F.round(
            F.col("purchase_rate_last_3_orders")
            - F.col("user_product_order_share"),
            4
        )
    )
    .select(
        "user_id",
        "product_id",
        "bought_in_last_order",
        "purchases_last_3_orders",
        "purchases_last_5_orders",
        "purchase_rate_last_3_orders",
        "purchase_rate_last_5_orders",
        "recent_purchase_momentum"
    )
)

display(user_product_momentum_features_df.limit(10))

user_id,product_id,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum
103738,1737,0,0,1,0.0,0.2,-0.0625
62689,8518,1,2,3,0.6667,0.6,0.6159
13741,29594,1,1,2,0.3333,0.4,0.0476
14528,1482,0,0,0,0.0,0.0,-0.0357
85771,8006,0,0,0,0.0,0.0,-0.0833
63280,49683,1,2,3,0.6667,0.6,0.2381
38337,23341,0,1,2,0.3333,0.4,0.0833
43747,49235,0,2,2,0.6667,0.4,0.4598
203734,1775,0,0,1,0.0,0.2,-0.1
73039,22164,1,2,2,0.6667,0.4,0.2667


### 6.4 — Create Purchase Streak Features

We measure how many consecutive orders contained each product. A long current streak can indicate that the product has become part of the customer's routine.

user_product_current_streak → consecutive recent orders containing the product

user_product_max_streak     → longest consecutive purchase streak in history

In [0]:
# Identify consecutive purchase sequences
product_streak_window = (
    Window
    .partitionBy("user_id", "product_id")
    .orderBy("order_number")
)

user_product_streak_base_df = (
    user_product_orders_df
    .withColumn(
        "purchase_sequence",
        F.row_number().over(product_streak_window)
    )
    .withColumn(
        "streak_group",
        F.col("order_number") - F.col("purchase_sequence")
    )
)

# Calculate each consecutive purchase streak
user_product_streaks_df = (
    user_product_streak_base_df
    .groupBy(
        "user_id",
        "product_id",
        "streak_group"
    )
    .agg(
        F.count("*").alias("streak_length"),
        F.max("order_number").alias("streak_end_order")
    )
)

# Maximum historical streak
max_streak_df = (
    user_product_streaks_df
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        F.max("streak_length").alias("user_product_max_streak")
    )
)

# Current streak: must end in the customer's last prior order
current_streak_df = (
    user_product_streaks_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_number"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "current_streak_value",
        F.when(
            F.col("streak_end_order") == F.col("target_order_number") - 1,
            F.col("streak_length")
        ).otherwise(0)
    )
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        F.max("current_streak_value")
        .alias("user_product_current_streak")
    )
)

user_product_streak_features_df = (
    max_streak_df
    .join(
        current_streak_df,
        on=["user_id", "product_id"],
        how="left"
    )
)

display(user_product_streak_features_df.limit(10))

user_id,product_id,user_product_max_streak,user_product_current_streak
1,196,10,10
1,10258,9,9
1,14084,1,0
2,79,1,0
2,4071,1,0
2,5869,1,0
2,7963,1,1
2,16797,1,0
2,21150,1,0
2,21227,1,0


### 6.5 — Combine Customer-Product Features

We combine all customer-product features into one table so each customer-product pair contains its loyalty, recency, momentum, and streak information.

In [0]:
all_customer_product_features_df = (
    customer_product_features_df
    .join(
        user_product_recency_features_df,
        on=["user_id", "product_id"],
        how="left"
    )
    .join(
        user_product_momentum_features_df,
        on=["user_id", "product_id"],
        how="left"
    )
    .join(
        user_product_streak_features_df,
        on=["user_id", "product_id"],
        how="left"
    )
)

display(all_customer_product_features_df.limit(10))

user_id,product_id,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak
126216,5450,36,0.9722,3.44,1,57,0.6102,3,1.6,1.88,0,1,1,0.3333,0.2,-0.2769,9,0
43505,44371,1,0.0,14.0,9,9,0.0222,37,null,null,0,0,0,0.0,0.0,-0.0222,1,0
30409,16797,3,0.6667,10.33,3,42,0.0714,1,19.5,0.05,1,1,1,0.3333,0.2,0.2619,1,1
94845,47626,3,0.6667,18.33,7,9,0.3333,1,1.0,1.0,1,3,3,1.0,0.6,0.6667,3,3
71641,38652,13,0.9231,11.15,8,40,0.2453,14,2.67,5.24,0,0,0,0.0,0.0,-0.2453,3,0
34885,13733,2,0.5,15.0,3,5,0.4,1,2.0,0.5,1,2,2,0.6667,0.4,0.2667,1,1
46943,5876,2,0.5,12.5,1,9,0.125,8,8.0,1.0,0,0,0,0.0,0.0,-0.125,1,0
7597,6135,1,0.0,3.0,22,22,0.0278,15,null,null,0,0,0,0.0,0.0,-0.0278,1,0
24192,44683,3,0.6667,10.33,2,20,0.0588,32,9.0,3.56,0,0,0,0.0,0.0,-0.0588,1,0
146255,812,2,0.5,7.0,21,22,0.0227,67,1.0,67.0,0,0,0,0.0,0.0,-0.0227,2,0


## 7. Category Affinity Features

### 7.1 — Create Customer Aisle Affinity

We measure how much of each customer's shopping history belongs to each aisle. A high value means the customer frequently buys products from that aisle.

In [0]:
historical_transactions_with_category_df = (
    historical_transactions_df
    .join(
        products_df.select(
            "product_id",
            "aisle_id",
            "department_id"
        ),
        on="product_id",
        how="left"
    )
)

customer_aisle_affinity_df = (
    historical_transactions_with_category_df
    .groupBy(
        "user_id",
        "aisle_id"
    )
    .agg(
        F.count("*").alias("customer_aisle_purchase_count")
    )
    .join(
        customer_features_df.select(
            "user_id",
            "customer_total_products"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "customer_aisle_affinity",
        F.round(
            F.col("customer_aisle_purchase_count")
            / F.col("customer_total_products"),
            4
        )
    )
    .select(
        "user_id",
        "aisle_id",
        "customer_aisle_purchase_count",
        "customer_aisle_affinity"
    )
)

display(customer_aisle_affinity_df.limit(10))

user_id,aisle_id,customer_aisle_purchase_count,customer_aisle_affinity
69984,81,2,0.0084
56022,91,17,0.0111
184355,86,2,0.022
133849,32,34,0.0342
113831,115,41,0.0869
133183,54,8,0.0307
114191,98,48,0.2759
195186,120,19,0.0691
149408,42,5,0.0275
28585,91,1,0.0152


### 7.2 — Create Customer Department Affinity

We measure how much of each customer's shopping history belongs to each department. This shows which broad product categories the customer prefers.

In [0]:
customer_department_affinity_df = (
    historical_transactions_with_category_df
    .groupBy(
        "user_id",
        "department_id"
    )
    .agg(
        F.count("*").alias("customer_department_purchase_count")
    )
    .join(
        customer_features_df.select(
            "user_id",
            "customer_total_products"
        ),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "customer_department_affinity",
        F.round(
            F.col("customer_department_purchase_count")
            / F.col("customer_total_products"),
            4
        )
    )
    .select(
        "user_id",
        "department_id",
        "customer_department_purchase_count",
        "customer_department_affinity"
    )
)

display(customer_department_affinity_df.limit(10))

user_id,department_id,customer_department_purchase_count,customer_department_affinity
30822,17,18,0.0381
31038,16,19,0.1152
113831,14,9,0.0191
39460,9,11,0.0133
114097,1,35,0.0873
133088,17,40,0.1434
80605,20,53,0.0457
40828,4,236,0.4126
166086,1,5,0.0962
175734,7,38,0.1169


### 7.3 — Map Category Affinity to Each Candidate Product

We attach the customer's aisle and department preferences to each candidate product. This shows how well each product matches the customer's usual category interests.

In [0]:
candidate_category_affinity_df = (
    candidate_products_df
    .join(
        all_product_features_df.select(
            "product_id",
            "aisle_id",
            "department_id"
        ),
        on="product_id",
        how="left"
    )
    .join(
        customer_aisle_affinity_df.select(
            "user_id",
            "aisle_id",
            "customer_aisle_affinity"
        ),
        on=["user_id", "aisle_id"],
        how="left"
    )
    .join(
        customer_department_affinity_df.select(
            "user_id",
            "department_id",
            "customer_department_affinity"
        ),
        on=["user_id", "department_id"],
        how="left"
    )
    .fillna({
        "customer_aisle_affinity": 0.0,
        "customer_department_affinity": 0.0
    })
)

display(candidate_category_affinity_df.limit(10))

user_id,department_id,aisle_id,product_id,target_order_id,customer_aisle_affinity,customer_department_affinity
166655,19,107,35561,765404,0.0144,0.1219
166655,19,50,46955,765404,0.0305,0.1219
166655,4,83,45007,765404,0.0786,0.2879
166655,4,24,19057,765404,0.1387,0.2879
166655,14,121,39560,765404,0.0209,0.0353
166655,1,34,12144,765404,0.0128,0.1163
166655,4,24,25715,765404,0.1387,0.2879
166655,12,106,45633,765404,0.0128,0.0128
166655,4,123,27966,765404,0.065,0.2879
166655,16,84,5785,765404,0.0168,0.1957


## 8. Temporal Compatibility Features

### 8.1 — Compare the Target Order with Customer and Product Habits

We compare the target order's day and hour with the customer's and product's usual shopping times. A closer match can indicate a higher probability of purchase.

customer_day_match       → 1 if the target order is on the customer's favorite day

product_day_match        → 1 if it is on the product's most common day

customer_hour_distance   → difference from the customer's usual shopping hour

product_hour_distance    → difference from the product's usual purchase hour

In [0]:
candidate_temporal_features_df = (
    candidate_products_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id",
            "target_order_dow",
            "target_order_hour"
        ),
        on=["user_id", "target_order_id"],
        how="inner"
    )
    .join(
        customer_routine_features_df.select(
            "user_id",
            "customer_preferred_dow",
            "customer_preferred_hour"
        ),
        on="user_id",
        how="left"
    )
    .join(
        product_temporal_features_df.select(
            "product_id",
            "product_preferred_dow",
            "product_preferred_hour"
        ),
        on="product_id",
        how="left"
    )
    .withColumn(
        "customer_day_match",
        F.when(
            F.col("target_order_dow") == F.col("customer_preferred_dow"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "product_day_match",
        F.when(
            F.col("target_order_dow") == F.col("product_preferred_dow"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "customer_hour_distance",
        F.abs(
            F.col("target_order_hour")
            - F.col("customer_preferred_hour")
        )
    )
    .withColumn(
        "product_hour_distance",
        F.abs(
            F.col("target_order_hour")
            - F.col("product_preferred_hour")
        )
    )
)

display(candidate_temporal_features_df.limit(10))

product_id,user_id,target_order_id,target_order_dow,target_order_hour,customer_preferred_dow,customer_preferred_hour,product_preferred_dow,product_preferred_hour,customer_day_match,product_day_match,customer_hour_distance,product_hour_distance
4920,201744,2027022,4,13,6,15,0,10,0,0,2,3
5134,172793,630046,0,11,1,21,0,9,0,1,10,2
21607,1059,2406692,1,16,1,15,1,11,1,1,1,5
39275,133849,713348,2,12,0,16,0,10,0,0,4,2
49519,65561,1064305,4,9,1,10,1,10,0,0,1,1
13870,11335,2634088,3,18,1,17,0,10,0,0,1,8
11568,202833,2912801,5,14,0,15,0,9,0,0,1,5
42450,192657,315899,5,4,1,17,0,14,0,0,13,10
47912,49413,877422,1,8,1,8,1,10,1,1,0,2
17758,96952,1804856,0,9,0,12,0,15,1,1,3,6


### 8.2 — Create Circular Time-Distance Features

We measure how close the target order is to the customer's and product's usual day and hour while accounting for the weekly and 24-hour cycles.

customer_day_distance             → distance from customer's usual shopping day

product_day_distance              → distance from product's usual purchase day

customer_hour_distance_circular   → distance from customer's usual hour

product_hour_distance_circular    → distance from product's usual hour

In [0]:
candidate_temporal_features_df = (
    candidate_temporal_features_df
    .withColumn(
        "customer_hour_distance_circular",
        F.least(
            F.col("customer_hour_distance"),
            24 - F.col("customer_hour_distance")
        )
    )
    .withColumn(
        "product_hour_distance_circular",
        F.least(
            F.col("product_hour_distance"),
            24 - F.col("product_hour_distance")
        )
    )
    .withColumn(
        "customer_day_distance",
        F.least(
            F.abs(
                F.col("target_order_dow")
                - F.col("customer_preferred_dow")
            ),
            7 - F.abs(
                F.col("target_order_dow")
                - F.col("customer_preferred_dow")
            )
        )
    )
    .withColumn(
        "product_day_distance",
        F.least(
            F.abs(
                F.col("target_order_dow")
                - F.col("product_preferred_dow")
            ),
            7 - F.abs(
                F.col("target_order_dow")
                - F.col("product_preferred_dow")
            )
        )
    )
)

display(
    candidate_temporal_features_df.select(
        "user_id",
        "product_id",
        "target_order_dow",
        "target_order_hour",
        "customer_preferred_dow",
        "customer_preferred_hour",
        "product_preferred_dow",
        "product_preferred_hour",
        "customer_day_distance",
        "product_day_distance",
        "customer_hour_distance_circular",
        "product_hour_distance_circular"
    ).limit(10)
)

user_id,product_id,target_order_dow,target_order_hour,customer_preferred_dow,customer_preferred_hour,product_preferred_dow,product_preferred_hour,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular
201744,4920,4,13,6,15,0,10,2,3,2,3
172793,5134,0,11,1,21,0,9,1,0,10,2
1059,21607,1,16,1,15,1,11,0,0,1,5
133849,39275,2,12,0,16,0,10,2,2,4,2
65561,49519,4,9,1,10,1,10,3,3,1,1
11335,13870,3,18,1,17,0,10,2,3,1,8
202833,11568,5,14,0,15,0,9,2,2,1,5
192657,42450,5,4,1,17,0,14,3,2,11,10
49413,47912,1,8,1,8,1,10,0,0,0,2
96952,17758,0,9,0,12,0,15,0,0,3,6


## 9. Basket Relationship Features

### 9.1 — Measure Category Continuity from the Last Basket

We check whether each candidate product belongs to an aisle or department that appeared in the customer's most recent historical basket. This captures short-term shopping context.

aisle_in_last_basket       → 1 if this product's aisle appeared in the last basket

department_in_last_basket  → 1 if this product's department appeared in the last basket

In [0]:
# Find each customer's most recent historical order
last_historical_order_df = (
    historical_orders_df
    .groupBy("user_id")
    .agg(
        F.max("order_number").alias("last_historical_order_number")
    )
)

# Extract the products and categories from that last basket
last_basket_categories_df = (
    historical_transactions_with_category_df
    .join(
        last_historical_order_df,
        on="user_id",
        how="inner"
    )
    .filter(
        F.col("order_number") == F.col("last_historical_order_number")
    )
    .select(
        "user_id",
        "aisle_id",
        "department_id"
    )
)

last_basket_aisles_df = (
    last_basket_categories_df
    .select("user_id", "aisle_id")
    .distinct()
    .withColumn("aisle_in_last_basket", F.lit(1))
)

last_basket_departments_df = (
    last_basket_categories_df
    .select("user_id", "department_id")
    .distinct()
    .withColumn("department_in_last_basket", F.lit(1))
)

candidate_basket_context_df = (
    candidate_category_affinity_df
    .join(
        last_basket_aisles_df,
        on=["user_id", "aisle_id"],
        how="left"
    )
    .join(
        last_basket_departments_df,
        on=["user_id", "department_id"],
        how="left"
    )
    .fillna({
        "aisle_in_last_basket": 0,
        "department_in_last_basket": 0
    })
)

display(
    candidate_basket_context_df.select(
        "user_id",
        "product_id",
        "aisle_id",
        "department_id",
        "aisle_in_last_basket",
        "department_in_last_basket"
    ).limit(10)
)

user_id,product_id,aisle_id,department_id,aisle_in_last_basket,department_in_last_basket
166655,35561,107,19,1,1
166655,46955,50,19,1,1
166655,45007,83,4,1,1
166655,19057,24,4,1,1
166655,39560,121,14,0,1
166655,12144,34,1,0,1
166655,25715,24,4,1,1
166655,45633,106,12,0,0
166655,27966,123,4,1,1
166655,5785,84,16,0,1


### 9.2 — Measure Customer Basket Stability

We compare the customer's two most recent historical baskets. A high overlap means the customer tends to repeat similar products from one order to the next.

close to 1 → the customer's recent baskets are very similar

close to 0 → the customer changes products a lot between orders

In [0]:
# Rank historical orders from most recent to oldest
recent_orders_ranked_df = (
    historical_orders_df
    .withColumn(
        "recent_order_rank",
        F.row_number().over(
            Window
            .partitionBy("user_id")
            .orderBy(F.desc("order_number"))
        )
    )
    .filter(F.col("recent_order_rank") <= 2)
    .select(
        "user_id",
        "order_id",
        "recent_order_rank"
    )
)

# Get products from the two most recent baskets
recent_basket_products_df = (
    historical_transactions_df
    .join(
        recent_orders_ranked_df,
        on=["user_id", "order_id"],
        how="inner"
    )
    .select(
        "user_id",
        "product_id",
        "recent_order_rank"
    )
    .distinct()
)

last_basket_products_df = (
    recent_basket_products_df
    .filter(F.col("recent_order_rank") == 1)
    .select("user_id", "product_id")
)

previous_basket_products_df = (
    recent_basket_products_df
    .filter(F.col("recent_order_rank") == 2)
    .select("user_id", "product_id")
)

# Basket sizes
last_basket_size_df = (
    last_basket_products_df
    .groupBy("user_id")
    .agg(
        F.count("*").alias("last_basket_unique_products")
    )
)

previous_basket_size_df = (
    previous_basket_products_df
    .groupBy("user_id")
    .agg(
        F.count("*").alias("previous_basket_unique_products")
    )
)

# Number of products appearing in both baskets
basket_overlap_df = (
    last_basket_products_df
    .join(
        previous_basket_products_df,
        on=["user_id", "product_id"],
        how="inner"
    )
    .groupBy("user_id")
    .agg(
        F.count("*").alias("repeated_products_between_last_two_baskets")
    )
)

# Jaccard similarity between the two baskets
customer_basket_stability_df = (
    last_basket_size_df
    .join(
        previous_basket_size_df,
        on="user_id",
        how="left"
    )
    .join(
        basket_overlap_df,
        on="user_id",
        how="left"
    )
    .fillna({
        "previous_basket_unique_products": 0,
        "repeated_products_between_last_two_baskets": 0
    })
    .withColumn(
        "customer_basket_stability",
        F.round(
            F.col("repeated_products_between_last_two_baskets")
            /
            (
                F.col("last_basket_unique_products")
                + F.col("previous_basket_unique_products")
                - F.col("repeated_products_between_last_two_baskets")
            ),
            4
        )
    )
)

display(customer_basket_stability_df.limit(10))

user_id,last_basket_unique_products,previous_basket_unique_products,repeated_products_between_last_two_baskets,customer_basket_stability
172959,6,14,4,0.25
84117,8,1,0,0.0
83143,3,4,0,0.0
185259,1,7,0,0.0
20185,27,25,12,0.3
194979,1,3,0,0.0
130096,11,22,1,0.0313
125295,12,20,2,0.0667
80517,3,1,1,0.3333
60305,18,12,5,0.2


### 9.3 — Create a Repeat-Stability Signal

We combine the customer's basket stability with whether the candidate product was bought in the last order. A high value indicates a routine customer repeating a recently purchased product.

0       → product was not in the last basket, or customer has no basket stability

high    → product was in the last basket and customer strongly repeats similar baskets

In [0]:
candidate_basket_behavior_df = (
    candidate_basket_context_df
    .join(
        customer_basket_stability_df.select(
            "user_id",
            "customer_basket_stability"
        ),
        on="user_id",
        how="left"
    )
    .join(
        all_customer_product_features_df.select(
            "user_id",
            "product_id",
            "bought_in_last_order"
        ),
        on=["user_id", "product_id"],
        how="left"
    )
    .fillna({
        "customer_basket_stability": 0.0,
        "bought_in_last_order": 0
    })
    .withColumn(
        "repeat_stability_signal",
        F.round(
            F.col("customer_basket_stability")
            * F.col("bought_in_last_order"),
            4
        )
    )
)

display(
    candidate_basket_behavior_df.select(
        "user_id",
        "product_id",
        "bought_in_last_order",
        "customer_basket_stability",
        "repeat_stability_signal",
        "aisle_in_last_basket",
        "department_in_last_basket"
    ).limit(10)
)

user_id,product_id,bought_in_last_order,customer_basket_stability,repeat_stability_signal,aisle_in_last_basket,department_in_last_basket
166655,5785,0,0.3171,0.0,0,1
166655,27966,0,0.3171,0.0,1,1
166655,45633,0,0.3171,0.0,0,0
166655,25715,0,0.3171,0.0,1,1
166655,12144,0,0.3171,0.0,0,1
166655,39560,0,0.3171,0.0,0,1
166655,19057,1,0.3171,0.3171,1,1
166655,45007,0,0.3171,0.0,1,1
166655,46955,0,0.3171,0.0,1,1
166655,35561,0,0.3171,0.0,1,1


### 9.4 — Create a Basket Context Level

We measure how closely each candidate product matches the customer's latest basket: exact product, same aisle, same department, or no recent connection.

last_basket_context_level

3 → exact product was in the last basket

2 → same aisle appeared

1 → same department appeared

0 → no recent basket connection

In [0]:
candidate_basket_behavior_df = (
    candidate_basket_behavior_df
    .withColumn(
        "last_basket_context_level",
        F.when(F.col("bought_in_last_order") == 1, 3)
         .when(F.col("aisle_in_last_basket") == 1, 2)
         .when(F.col("department_in_last_basket") == 1, 1)
         .otherwise(0)
    )
    .withColumn(
        "stability_weighted_context",
        F.round(
            (F.col("last_basket_context_level") / F.lit(3.0))
            * F.col("customer_basket_stability"),
            4
        )
    )
)

display(
    candidate_basket_behavior_df.select(
        "user_id",
        "product_id",
        "bought_in_last_order",
        "aisle_in_last_basket",
        "department_in_last_basket",
        "last_basket_context_level",
        "customer_basket_stability",
        "stability_weighted_context"
    ).limit(10)
)

user_id,product_id,bought_in_last_order,aisle_in_last_basket,department_in_last_basket,last_basket_context_level,customer_basket_stability,stability_weighted_context
156804,21376,1,1,1,3,0.1154,0.1154
28551,21376,0,0,0,0,0.0,0.0
42062,21376,1,1,1,3,0.1667,0.1667
31919,21376,0,1,1,2,0.0833,0.0555
199248,4366,0,0,1,1,0.0,0.0
144409,21376,1,1,1,3,0.1,0.1
151819,37787,0,0,0,0,0.2667,0.0
36663,21376,0,1,1,2,0.125,0.0833
87084,17149,0,0,1,1,0.28,0.0933
23797,21376,1,1,1,3,0.0189,0.0189


## 10. Target Order Context Features

### 10.1 — Compare the Target Order with the Customer's Usual Rhythm

We compare the time since the customer's last order with their usual ordering interval. This helps identify whether the next order happens earlier, normally, or later than expected.

order_timing_ratio ≈ 1  → customer ordered around their usual time

order_timing_ratio < 1  → customer ordered earlier than usual

order_timing_ratio > 1  → customer ordered later than usual

In [0]:
customer_order_context_df = (
    target_orders_df
    .join(
        customer_rhythm_features_df.select(
            "user_id",
            "customer_avg_days_between_orders"
        ),
        on="user_id",
        how="left"
    )
    .withColumn(
        "order_timing_deviation_days",
        F.round(
            F.col("target_days_since_prior_order")
            - F.col("customer_avg_days_between_orders"),
            2
        )
    )
    .withColumn(
        "order_timing_ratio",
        F.when(
            F.col("customer_avg_days_between_orders") > 0,
            F.round(
                F.col("target_days_since_prior_order")
                / F.col("customer_avg_days_between_orders"),
                2
            )
        )
    )
    .select(
        "user_id",
        "target_order_id",
        "target_days_since_prior_order",
        "customer_avg_days_between_orders",
        "order_timing_deviation_days",
        "order_timing_ratio"
    )
)

display(customer_order_context_df.limit(10))

user_id,target_order_id,target_days_since_prior_order,customer_avg_days_between_orders,order_timing_deviation_days,order_timing_ratio
1,1187899,14.0,19.56,-5.56,0.72
2,1492625,30.0,15.23,14.77,1.97
5,2196797,6.0,13.33,-7.33,0.45
7,525192,6.0,10.68,-4.68,0.56
8,880375,10.0,30.0,-20.0,0.33
9,1094988,30.0,18.0,12.0,1.67
10,1822501,30.0,19.75,10.25,1.52
13,1827621,8.0,7.64,0.36,1.05
14,2316178,11.0,22.08,-11.08,0.5
17,2180313,30.0,7.44,22.56,4.03


For example:

user 1 → ratio 0.72

means the target order happened earlier than this customer's usual rhythm.

user 17 → ratio 4.03

means the target order happened much later than usual.

### 10.2 — Estimate Expected Basket Size

We compare the customer's recent basket behavior with their historical average to estimate whether the target order may be larger or smaller than usual.

ratio > 1 → recent baskets are larger than usual

ratio < 1 → recent baskets are smaller than usual

ratio ≈ 1 → stable basket size

In [0]:
customer_expected_basket_df = (
    all_customer_features_df
    .select(
        "user_id",
        "customer_avg_basket_size",
        "customer_last_basket_size",
        "customer_avg_last_3_basket_size",
        "customer_basket_size_trend"
    )
    .withColumn(
        "expected_basket_size",
        F.round(
            (
                F.col("customer_avg_basket_size")
                + F.col("customer_avg_last_3_basket_size")
            ) / 2,
            2
        )
    )
    .withColumn(
        "recent_vs_usual_basket_ratio",
        F.when(
            F.col("customer_avg_basket_size") > 0,
            F.round(
                F.col("customer_avg_last_3_basket_size")
                / F.col("customer_avg_basket_size"),
                2
            )
        )
    )
)

display(customer_expected_basket_df.limit(10))

user_id,customer_avg_basket_size,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,expected_basket_size,recent_vs_usual_basket_ratio
100296,14.87,27,17.0,2.13,15.93,1.14
32537,4.98,6,6.33,1.35,5.66,1.27
121916,9.24,9,12.33,3.09,10.79,1.33
119458,11.67,12,11.67,0.0,11.67,1.0
54861,7.77,11,7.0,-0.77,7.39,0.9
63226,10.97,19,11.67,0.7,11.32,1.06
68579,3.0,5,3.33,0.33,3.17,1.11
34174,16.0,10,16.0,0.0,16.0,1.0
188741,9.0,11,8.33,-0.67,8.67,0.93
89274,4.75,2,6.0,1.25,5.38,1.26


### 10.3 — Combine Target Order Context Features

We combine the customer's order timing and expected basket size into one table that describes the context of the order we want to predict.

In [0]:
all_order_context_features_df = (
    customer_order_context_df
    .join(
        customer_expected_basket_df.select(
            "user_id",
            "expected_basket_size",
            "recent_vs_usual_basket_ratio"
        ),
        on="user_id",
        how="left"
    )
)

display(all_order_context_features_df.limit(10))

user_id,target_order_id,target_days_since_prior_order,customer_avg_days_between_orders,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio
1,1187899,14.0,19.56,-5.56,0.72,6.45,1.19
2,1492625,30.0,15.23,14.77,1.97,14.3,1.05
5,2196797,6.0,13.33,-7.33,0.45,8.96,0.94
7,525192,6.0,10.68,-4.68,0.56,10.15,0.97
8,880375,10.0,30.0,-20.0,0.33,16.33,1.0
9,1094988,30.0,18.0,12.0,1.67,25.33,1.0
10,1822501,30.0,19.75,10.25,1.52,29.64,1.07
13,1827621,8.0,7.64,0.36,1.05,7.54,1.23
14,2316178,11.0,22.08,-11.08,0.5,14.58,0.8
17,2180313,30.0,7.44,22.56,4.03,8.34,1.27


## 11. Assemble the Modeling Dataset

### 11.1 — Combine All Engineered Features

We combine the customer, product, customer-product, category, temporal, basket, and order-context features into one dataset. Each row represents one candidate product for one customer's target order.

Customer

        ×

Candidate Product

        ×

Target Order

         ↓

All engineered features

         +

target_reordered

In [0]:
model_features_df = (
    model_base_df

    # Customer features
    .join(
        all_customer_features_df,
        on="user_id",
        how="left"
    )

    # Product features
    .join(
        all_product_features_df,
        on="product_id",
        how="left"
    )

    # Customer-product features
    .join(
        all_customer_product_features_df,
        on=["user_id", "product_id"],
        how="left"
    )

    # Category + basket-context features
    .join(
        candidate_basket_behavior_df.select(
            "user_id",
            "product_id",
            "target_order_id",
            "customer_aisle_affinity",
            "customer_department_affinity",
            "aisle_in_last_basket",
            "department_in_last_basket",
            "customer_basket_stability",
            "repeat_stability_signal",
            "last_basket_context_level",
            "stability_weighted_context"
        ),
        on=["user_id", "product_id", "target_order_id"],
        how="left"
    )

    # Temporal compatibility features
    .join(
        candidate_temporal_features_df.select(
            "user_id",
            "product_id",
            "target_order_id",
            "target_order_dow",
            "target_order_hour",
            "customer_day_match",
            "product_day_match",
            "customer_day_distance",
            "product_day_distance",
            "customer_hour_distance_circular",
            "product_hour_distance_circular"
        ),
        on=["user_id", "product_id", "target_order_id"],
        how="left"
    )

    # Target-order context
    .join(
        all_order_context_features_df.select(
            "user_id",
            "target_order_id",
            "target_days_since_prior_order",
            "order_timing_deviation_days",
            "order_timing_ratio",
            "expected_basket_size",
            "recent_vs_usual_basket_ratio"
        ),
        on=["user_id", "target_order_id"],
        how="left"
    )

    # Target order number
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id",
            "target_order_number"
        ),
        on=["user_id", "target_order_id"],
        how="left"
    )
)

display(model_features_df.limit(10))

user_id,target_order_id,product_id,target_reordered,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,customer_aisle_affinity,customer_department_affinity,aisle_in_last_basket,department_in_last_basket,customer_basket_stability,repeat_stability_signal,last_basket_context_level,stability_weighted_context,target_order_dow,target_order_hour,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number
96952,1804856,17758,0,57,560,207,9.82,0.6304,6.41,3.9,11.44,7,10,10.0,0.18,0,0.2105,12,0.1754,5959,2570,0.5687,8.47,Strawberry Rhubarb Yoghurt,120,16,2.32,0.002911,0,0.1834,15,0.0826,4,0.75,7.0,2,9,0.0702,49,2.33,21.03,0,0,0,0.0,0.0,-0.0702,3,0,0.0179,0.1196,0,0,0.0556,0.0,0,0.0,0,9,1,1,0,0,3,6,5.0,-1.41,0.78,9.91,1.02,58
172793,630046,5134,0,99,1306,269,13.19,0.794,3.22,1.23,17.85,7,8,12.33,-0.86,1,0.3838,21,0.2424,7931,3083,0.6113,11.25,Organic Thompson Seedless Raisins,117,19,2.57,0.003874,0,0.2035,9,0.082,13,0.9231,6.0,10,92,0.1313,8,6.83,1.17,0,0,0,0.0,0.0,-0.1313,2,0,0.0168,0.0551,0,0,0.1,0.0,0,0.0,0,11,0,1,1,0,10,2,1.0,-2.22,0.31,12.76,0.93,100
94143,590209,5785,0,27,517,113,19.15,0.7814,13.0,7.86,13.44,7,25,19.0,-0.15,0,0.1852,11,0.2963,30072,5913,0.8034,5.74,Organic Reduced Fat 2% Milk,84,16,5.09,0.014688,1,0.1938,10,0.093,8,0.875,11.63,1,25,0.2963,3,3.43,0.87,0,1,1,0.3333,0.2,0.037,4,0,0.0368,0.2495,0,1,0.0909,0.0,1,0.0303,6,12,0,0,1,2,1,2,8.0,-5.0,0.62,19.08,0.99,28
30564,1258432,24852,1,52,740,242,14.23,0.673,5.92,5.8,13.9,7,11,16.0,1.77,0,0.3462,9,0.1154,300764,46964,0.8439,4.89,Banana,24,4,6.4,0.146902,0,0.2038,10,0.086,19,0.9474,7.16,1,50,0.3654,3,2.72,1.1,0,1,2,0.3333,0.4,-0.0321,6,0,0.0811,0.3162,1,1,0.0,0.0,2,0.0,2,17,0,0,2,2,8,7,9.0,3.08,1.52,15.12,1.12,53
16592,805129,40706,0,47,475,125,10.11,0.7368,7.24,7.18,14.66,7,9,15.0,4.89,2,0.1915,13,0.1277,53982,18577,0.6559,9.43,Organic Grape Tomatoes,123,4,2.91,0.026366,0,0.2333,15,0.0866,2,0.5,3.5,31,35,0.0426,13,4.0,3.25,0,0,0,0.0,0.0,-0.0426,1,0,0.12,0.7032,1,1,0.1379,0.0,2,0.0919,0,22,0,1,2,0,9,7,13.0,5.76,1.8,12.56,1.48,48
117752,3067566,38278,1,99,547,83,5.53,0.8483,2.98,1.71,13.05,7,12,6.33,0.8,0,0.1717,13,0.1111,158,58,0.6329,5.99,Pink Beans,59,15,2.72,7.7E-5,3,0.1962,14,0.1139,23,0.9565,4.83,26,99,0.2323,1,3.32,0.3,1,1,1,0.3333,0.2,0.101,2,1,0.2431,0.3455,1,1,0.0,0.0,3,0.0,4,15,0,0,3,1,2,1,6.0,3.02,2.01,5.93,1.14,100
148452,1548499,6870,0,19,176,112,9.26,0.3636,14.44,10.4,13.89,5,16,12.0,2.74,0,0.2632,16,0.2632,318,211,0.3365,8.7,Fresh Pressed Coconut Oil,19,13,1.51,1.55E-4,1,0.1824,10,0.1069,1,0.0,7.0,17,17,0.0526,3,null,null,0,1,1,0.3333,0.2,0.2807,1,0,0.0114,0.0625,0,1,0.0,0.0,1,0.0,6,14,0,0,1,2,2,4,2.0,-12.44,0.14,10.63,1.3,20
176946,1230295,33279,0,

### 11.2 — Validate Modeling Dataset Row Uniqueness

We check that each customer-product-target order combination appears only once and that combining the features did not create duplicate rows.

In [0]:
model_dataset_check_df = (
    model_features_df
    .agg(
        F.count("*").alias("total_rows"),

        F.countDistinct(
            F.struct(
                "user_id",
                "product_id",
                "target_order_id"
            )
        ).alias("unique_customer_product_orders")
    )
    .withColumn(
        "duplicate_rows",
        F.col("total_rows")
        - F.col("unique_customer_product_orders")
    )
)

display(model_dataset_check_df)

total_rows,unique_customer_product_orders,duplicate_rows
8474661,8474661,0


### 11.3 — Check Missing Values in Engineered Features

We count missing values in the modeling dataset so we can identify which features need cleaning or default values before training the model.

In [0]:
missing_values_df = (
    model_features_df
    .select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in model_features_df.columns
    ])
)

display(missing_values_df)

user_id,target_order_id,product_id,target_reordered,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,customer_aisle_affinity,customer_department_affinity,aisle_in_last_basket,department_in_last_basket,customer_basket_stability,repeat_stability_signal,last_basket_context_level,stability_weighted_context,target_order_dow,target_order_hour,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5083521,5083521,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,135,0,0,0


### 11.4 — Handle Missing Feature Values

Some products were purchased only once, so no reorder cycle can be calculated. We keep these observations, create indicators showing whether the information exists, and replace the remaining missing numerical values with safe defaults.

In [0]:
model_features_clean_df = (
    model_features_df

    # Indicator: does this customer-product pair have enough history
    # to calculate a reorder cycle?
    .withColumn(
        "has_reorder_cycle",
        F.when(
            F.col("user_product_avg_order_gap").isNotNull(),
            1
        ).otherwise(0)
    )

    # Indicator: is the order timing ratio available?
    .withColumn(
        "has_order_timing_ratio",
        F.when(
            F.col("order_timing_ratio").isNotNull(),
            1
        ).otherwise(0)
    )

    # Replace remaining numerical nulls
    .fillna({
        "user_product_avg_order_gap": 0.0,
        "user_product_due_score": 0.0,
        "order_timing_ratio": 1.0
    })
)

display(
    model_features_clean_df.select(
        "user_id",
        "product_id",
        "user_product_avg_order_gap",
        "user_product_due_score",
        "has_reorder_cycle",
        "order_timing_ratio",
        "has_order_timing_ratio"
    ).limit(10)
)

user_id,product_id,user_product_avg_order_gap,user_product_due_score,has_reorder_cycle,order_timing_ratio,has_order_timing_ratio
96952,17758,2.33,21.03,1,0.78,1
172793,5134,6.83,1.17,1,0.31,1
94143,5785,3.43,0.87,1,0.62,1
30564,24852,2.72,1.1,1,1.52,1
16592,40706,4.0,3.25,1,1.8,1
117752,38278,3.32,0.3,1,2.01,1
148452,6870,0.0,0.0,0,0.14,1
176946,33279,6.0,0.5,1,0.14,1
146921,23909,1.5,0.67,1,0.63,1
102020,26252,6.67,3.0,1,1.43,1


### 11.5 — Verify Missing Values After Cleaning

We check the dataset again to confirm that all missing feature values have been handled before model training.

In [0]:
remaining_missing_values_df = (
    model_features_clean_df
    .select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in model_features_clean_df.columns
    ])
)

display(remaining_missing_values_df)

user_id,target_order_id,product_id,target_reordered,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,customer_aisle_affinity,customer_department_affinity,aisle_in_last_basket,department_in_last_basket,customer_basket_stability,repeat_stability_signal,last_basket_context_level,stability_weighted_context,target_order_dow,target_order_hour,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number,has_reorder_cycle,has_order_timing_ratio
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 11.6 — Validate Historical and Target Order Separation

We verify that all features come from orders before the target order. This confirms that no target-order information leaked into the feature engineering process.

In [0]:
historical_max_order_df = (
    historical_orders_df
    .groupBy("user_id")
    .agg(
        F.max("order_number").alias("max_historical_order_number")
    )
)

leakage_check_df = (
    target_orders_df
    .select(
        "user_id",
        "target_order_number"
    )
    .join(
        historical_max_order_df,
        on="user_id",
        how="inner"
    )
    .withColumn(
        "is_valid",
        F.when(
            F.col("max_historical_order_number")
            < F.col("target_order_number"),
            1
        ).otherwise(0)
    )
    .agg(
        F.count("*").alias("customers_checked"),
        F.sum(
            F.when(F.col("is_valid") == 0, 1).otherwise(0)
        ).alias("leakage_violations")
    )
)

display(leakage_check_df)

customers_checked,leakage_violations
131209,0


### 11.7 — Define the Prediction-Time Assumption

We assume predictions are generated when the customer starts their next shopping session. Therefore, the target order's day, hour, and time since the previous order are known at prediction time and can safely be used as features.

### 11.8 — Final Feature Dataset Validation

We summarize the final modeling dataset to confirm its size, number of customers, products, features, and target distribution before saving it.

In [0]:
final_dataset_summary_df = (
    model_features_clean_df
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.countDistinct("product_id").alias("unique_products"),
        F.sum("target_reordered").alias("reordered_products")
    )
    .withColumn(
        "non_reordered_products",
        F.col("total_rows") - F.col("reordered_products")
    )
    .withColumn(
        "reorder_percentage",
        F.round(
            F.col("reordered_products")
            / F.col("total_rows") * 100,
            2
        )
    )
    .withColumn(
        "total_columns",
        F.lit(len(model_features_clean_df.columns))
    )
)

display(final_dataset_summary_df)

total_rows,unique_customers,unique_products,reordered_products,non_reordered_products,reorder_percentage,total_columns
8474661,131209,49468,828824,7645837,9.78,74


### 11.9 — Save the Final Feature Dataset

We save the cleaned modeling dataset as a Delta table so it can be reused directly for machine learning without recalculating all features.

In [0]:
# Create the ML schema if it does not already exist
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.ml_data
""")

# Save the final feature dataset
(
    model_features_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ml_data.reorder_features")
)

print("Feature dataset saved successfully:")
print("workspace.ml_data.reorder_features")

Feature dataset saved successfully:
workspace.ml_data.reorder_features
